# مقدمه‌ای بر مهندسی پرامپت
مهندسی پرامپت فرایند طراحی و بهینه‌سازی پرامپت‌ها برای وظایف پردازش زبان طبیعی است. این فرایند شامل انتخاب پرامپت‌های مناسب، تنظیم پارامترهای آن‌ها و ارزیابی عملکردشان می‌شود. مهندسی پرامپت برای دستیابی به دقت و کارایی بالا در مدل‌های NLP حیاتی است. در این بخش، مبانی مهندسی پرامپت را با استفاده از مدل‌های OpenAI برای کاوش بررسی خواهیم کرد.


### تمرین ۱: توکنیزه کردن
بررسی توکنیزه کردن با استفاده از tiktoken، یک توکنیزر سریع و منبع‌باز از OpenAI
برای مثال‌های بیشتر، به [OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_count_tokens_with_tiktoken.ipynb?WT.mc_id=academic-105485-koreyst) مراجعه کنید.


In [ ]:
# EXERCISE:
# 1. Run the exercise as is first
# 2. Change the text to any prompt input you want to use & re-run to see tokens

import tiktoken

# Define the prompt you want tokenized
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

# Set the model you want encoding for
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode the text - gives you the tokens in integer form
tokens = encoding.encode(text)
print(tokens);

# Decode the integers to see what the text versions look like
[encoding.decode_single_token_bytes(token) for token in tokens]


### تمرین ۲: اعتبارسنجی تنظیم کلید مدل‌های Microsoft Foundry

> **توجه:** GitHub Models تا پایان ژوئیه ۲۰۲۶ بازنشسته خواهد شد. این تمرین به جای آن از [مدل‌های Microsoft Foundry](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst) استفاده می‌کند که همان کاتالوگ مدل رایگان برای آزمایش و تجربه SDK استنتاج هوش مصنوعی Azure را ارائه می‌دهد.

کد زیر را اجرا کنید تا بررسی شود که نقطه پایانی مدل‌های Microsoft Foundry شما به درستی تنظیم شده است. این کد فقط یک درخواست ساده اولیه را امتحان می‌کند و تکمیل آن را اعتبارسنجی می‌کند. ورودی `oh say can you see` باید چیزی در حدود `by the dawn's early light..` تکمیل شود.


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

model_name = "gpt-4o-mini"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

def get_completion(prompt, client, model_name, temperature=1.0, max_tokens=1000, top_p=1.0):
    response = client.complete(
        messages=[
            {
                "role": "system",
                "content": "You are a helpful assistant.",
            },
            {
                "role": "user",
                "content": prompt,
            },
        ],
        model=model_name,
        temperature=temperature,
        max_tokens=max_tokens,
        top_p=top_p
    )
    return response.choices[0].message.content

## ---------- Call the helper method

### 1. Set primary content or prompt text
text = f"""
oh say can you see
"""

### 2. Use that in the prompt template below
prompt = f"""
```{text}```
"""

## 3. Run the prompt
response = get_completion(prompt, client, model_name)
print(response)


That line is the opening lyric of "The Star-Spangled Banner," the national anthem of the United States, written by Francis Scott Key. If you'd like more information or analysis, feel free to ask!


### تمرین ۳: ساختگی‌ها
بررسی کنید چه اتفاقی می‌افتد وقتی از مدل زبان بزرگ (LLM) می‌خواهید تکمیل‌هایی برای یک پرسش درباره موضوعی که ممکن است وجود نداشته باشد، یا درباره موضوعاتی که ممکن است ندانسته باشد چون خارج از داده‌های پیش‌آموزشی آن بوده (جدیدتر)، ارائه دهد. ببینید چگونه پاسخ تغییر می‌کند اگر پرسش متفاوتی را امتحان کنید، یا مدل متفاوتی را به کار ببرید.


In [ ]:

## Set the text for simple prompt or primary content
## Prompt shows a template format with text in it - add cues, commands etc if needed
## Run the completion 
text = f"""
generate a lesson plan on the Martian War of 2076.
"""

prompt = f"""
```{text}```
"""

response = get_completion(prompt, client, model_name)
print(response)

### تمرین ۴: مبتنی بر دستورالعمل  
از متغیر "text" برای تعیین محتوای اصلی استفاده کنید  
و از متغیر "prompt" برای ارائه یک دستورالعمل مرتبط با آن محتوای اصلی استفاده کنید.  

در اینجا از مدل می‌خواهیم متن را برای یک دانش‌آموز کلاس دوم خلاصه کند  


In [4]:
# Test Example
# https://platform.openai.com/playground/p/default-summarize

## Example text
text = f"""
Jupiter is the fifth planet from the Sun and the \
largest in the Solar System. It is a gas giant with \
a mass one-thousandth that of the Sun, but two-and-a-half \
times that of all the other planets in the Solar System combined. \
Jupiter is one of the brightest objects visible to the naked eye \
in the night sky, and has been known to ancient civilizations since \
before recorded history. It is named after the Roman god Jupiter.[19] \
When viewed from Earth, Jupiter can be bright enough for its reflected \
light to cast visible shadows,[20] and is on average the third-brightest \
natural object in the night sky after the Moon and Venus.
"""

## Set the prompt
prompt = f"""
Summarize content you are provided with for a second-grade student.
```{text}```
"""

## Run the prompt
response = get_completion(prompt, client, model_name)
print(response)

Jupiter is the fifth planet from the Sun and the biggest one in our Solar System. It's made of gas and is much bigger than all the other planets put together! You can see Jupiter in the night sky because it's very bright. People have noticed it for a really long time and named it after a Roman god.


### تمرین ۵: درخواست پیچیده  
یک درخواستی را امتحان کنید که شامل پیام‌های سیستم، کاربر و دستیار باشد  
سیستم زمینه دستیار را تنظیم می‌کند  
پیام‌های کاربر و دستیار زمینه گفتگوی چند مرحله‌ای را فراهم می‌کنند  

دقت کنید که چگونه شخصیت دستیار در زمینه سیستم به "طعنه‌آمیز" تنظیم شده است.  
سعی کنید از زمینه شخصیت متفاوتی استفاده کنید. یا یک سری پیام‌های ورودی/خروجی متفاوت را امتحان کنید  


In [5]:
response = client.complete(
    model=model_name,
    messages=[
        {"role": "system", "content": "You are a sarcastic assistant."},
        {"role": "user", "content": "Who won the world series in 2020?"},
        {"role": "assistant", "content": "Who do you think won? The Los Angeles Dodgers of course."},
        {"role": "user", "content": "Where was it played?"}
    ]
)
print(response.choices[0].message.content)

Oh, you mean the famous 2020 World Series that wasn’t in a regular location? That was the year they played in the glamorous Arlington, Texas, at Globe Life Field.


### تمرین: شهود خود را کاوش کنید
مثال‌های بالا الگوهایی به شما می‌دهند که می‌توانید برای ایجاد درخواست‌های جدید (ساده، پیچیده، دستوری و غیره) استفاده کنید - تلاش کنید تمرین‌های دیگری ایجاد کنید تا برخی از ایده‌های دیگری که در موردشان صحبت کردیم مثل مثال‌ها، نشانه‌ها و موارد دیگر را کاوش کنید.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
